# Module 01: Storage Theory, ACID & Relational Fundamentals

Interactive lab exploring flat-file storage, ACID transactions, relational algebra, and disk page models.


In [ ]:
import time
print('Environment initialized.')


## 1. The $O(N)$ Unindexed File Scan Bottleneck

Compare linear scan on raw text files vs structured lookups.


In [ ]:
records = [{'id': i, 'user': f'user_{i}', 'val': i * 10} for i in range(10000)]
start = time.perf_counter()
# Linear search
match = next((r for r in records if r['id'] == 9999), None)
elapsed = time.perf_counter() - start
print(f'Found: {match} in {elapsed*1000:.3f} ms')


## 2. In-Memory Hash Index: O(1) Fast-Path Lookups

Creating an index eliminates linear scanning.


In [ ]:
index = {r['id']: r for r in records}
start = time.perf_counter()
fast_match = index.get(9999)
fast_elapsed = time.perf_counter() - start
print(f'Indexed lookup: {fast_match} in {fast_elapsed*1000:.4f} ms')
print(f'Speedup: {elapsed / fast_elapsed:.1f}x faster!')


## 3. ACID Atomicity & Write-Ahead Logging (WAL)

Simulating a transaction log that survives midway failure.


In [ ]:
wal_entries = []
def log_txn_begin(tx_id):
    wal_entries.append(f'BEGIN {tx_id}')

def log_txn_commit(tx_id):
    wal_entries.append(f'COMMIT {tx_id}')

log_txn_begin('tx_101')
wal_entries.append('INSERT INTO users VALUES (1, "Alice")')
log_txn_commit('tx_101')
print('WAL entries:', wal_entries)


## 4. Crash Recovery Simulation

Replaying committed transactions and discarding aborted operations.


In [ ]:
log_txn_begin('tx_102')
wal_entries.append('INSERT INTO users VALUES (2, "Bob")')
# Crash occurs before COMMIT!

committed = set()
for entry in wal_entries:
    if entry.startswith('COMMIT'):
        committed.add(entry.split()[1])

print('Committed Txns successfully recovered:', committed)
print('Discarded incomplete transactions: tx_102')


## 5. Relational Algebra: Selection (sigma) and Projection (pi)

Pure algebraic operations on tuple streams.


In [ ]:
dataset = [
    {'name': 'Alice', 'role': 'Admin', 'salary': 120000},
    {'name': 'Bob', 'role': 'Dev', 'salary': 95000},
    {'name': 'Carol', 'role': 'Dev', 'salary': 105000},
]
# Selection (salary > 100000)
selected = [row for row in dataset if row['salary'] > 100000]
# Projection (name, role)
projected = [{'name': r['name'], 'role': r['role']} for r in selected]
print('Selection + Projection result:', projected)


## 6. Normalization: Eliminating Transitive Dependencies (3NF)

Decomposing redundant columns to avoid update anomalies.


In [ ]:
denormalized_orders = [
    {'order_id': 1, 'cust_id': 10, 'cust_city': 'New York', 'amount': 150.0},
    {'order_id': 2, 'cust_id': 10, 'cust_city': 'New York', 'amount': 220.0},
]
# Normalized into two relations
customers = {10: {'city': 'New York'}}
orders = [{'order_id': 1, 'cust_id': 10, 'amount': 150.0}, {'order_id': 2, 'cust_id': 10, 'amount': 220.0}]
print('Normalized Customers:', customers)
print('Normalized Orders:', orders)


## Summary & Key Takeaways

1. Flat-file storage lacks indexing, leading to $O(N)$ query degradation.
2. WAL logs guarantee Atomicity and Durability across sudden power loss.
3. Normalization minimizes data duplication and prevents anomaly hazards.
